# Argument Binding, Defaults, and Unpacking Without Hand-Waving
Function signatures are where Python's friendly surface meets a very exact binding algorithm. This notebook makes that algorithm visible so defaults, *args, **kwargs, and unpacking stop feeling like special-case tricks.

When people struggle with **Argument Binding, Defaults, and Unpacking Without Hand-Waving**, the struggle is often not about syntax. It is usually about trying to reason from surface appearance alone. Python rewards a deeper habit: do not ask only what the code *looks like*; ask what objects exist, what names or attributes point to them, what bytecode-level actions are likely happening, and which runtime protocol is being used.

That habit is what turns memorized rules into real understanding. It also makes interviews easier, because you stop giving slogan answers and start giving mechanism-based answers.


- Understand how Python matches arguments to parameters
- See why mutable defaults are dangerous
- Read signatures more confidently
- Use unpacking without guesswork


Default parameter objects are created once at function definition time, not once per call. That one detail explains the famous mutable-default bug better than memorized warnings ever will.


In [1]:
def add_item(value, bucket=[]):
    bucket.append(value)
    return bucket

print(add_item(1))
print(add_item(2))
print(add_item.__defaults__)
print(id(add_item.__defaults__[0]))


[1]
[1, 2]
([1, 2],)
2618843892608


Disassembly is helpful here because it shows that defaults live with the function object and that calls are runtime operations where arguments are prepared and then sent into the function.


In [2]:
import dis

def sample(a, b=10):
    return a + b

dis.dis(sample)


  3           0 RESUME                   0

  4           2 LOAD_FAST                0 (a)
              4 LOAD_FAST                1 (b)
              6 BINARY_OP                0 (+)
             10 RETURN_VALUE


Python follows clear rules when mapping call arguments to parameters.

That is why mutable defaults persist across calls.

They also help avoid ambiguity when many optional values exist.

It is especially useful when values already live in a tuple, list, or dict.


The same function can often be called in multiple clear ways.


In [3]:
def connect(host, port, timeout=30):
    return host, port, timeout

print(connect("localhost", 5432))
print(connect(host="localhost", port=5432, timeout=5))


('localhost', 5432, 30)
('localhost', 5432, 5)


They gather extra positional and keyword inputs.


In [4]:
def show(*args, **kwargs):
    print(args)
    print(kwargs)

show(1, 2, 3, name="Ada", active=True)


(1, 2, 3)
{'name': 'Ada', 'active': True}


Stored values can be spread directly into parameters.


In [5]:
def total(a, b, c):
    return a + b + c

values = (1, 2, 3)
config = {"a": 4, "b": 5, "c": 6}
print(total(*values))
print(total(**config))


6
15


This is not only a classroom topic. **Argument Binding, Defaults, and Unpacking Without Hand-Waving** usually appears in real projects as a debugging problem, a design choice, a performance tradeoff, or a readability decision. In other words, this topic matters most when the codebase gets large enough that vague intuition stops being reliable.

That is why the low-level view is useful. It gives you something firmer than guesswork when code starts behaving in surprising ways.


- Using mutable defaults casually
- Treating *args and **kwargs as magic rather than collected values
- Reading signatures too quickly and missing how arguments really bind


- Why are mutable default arguments dangerous?
- What do *args and **kwargs actually receive?
- What happens when unpacking collides with explicit keyword arguments?


- Argument binding follows real rules.
- Default objects can persist.
- Unpacking is still just argument passing.


If this notebook did its job, **Argument Binding, Defaults, and Unpacking Without Hand-Waving** should now feel less like a bag of special rules and more like a natural consequence of Python's runtime model. That is the standard we want: not just recall, but a calm explanation of what the interpreter is seeing and why the behavior follows from that.


A better way to understand Argument Binding, Defaults, and Unpacking Without Hand-Waving is to keep moving back and forth between three views of the same code.

The first view is the human view. This is what the code is trying to say. The second view is the object view. In that view, we ask which Python objects exist, which names refer to them, and which objects are being created, reused, mutated, or discarded. The third view is the interpreter view. In that view, we ask what protocol is being used, what bytecode is being executed, and which runtime rules are controlling the result.

Many learners stop after the first view. That is enough to copy code, but it is usually not enough to explain surprising behavior. Deep understanding starts when these three views become one mental movement instead of three unrelated topics.


There is also a fourth habit that helps a lot: failure-based thinking.

For Argument Binding, Defaults, and Unpacking Without Hand-Waving, ask yourself what normally goes wrong when the idea is only half-understood. Does someone confuse identity with equality? Do they expect copying where only rebinding happened? Do they expect a fresh iterator when they really have an exhausted one? Do they trust a default value that is secretly shared? Do they think a class attribute is per-instance state? Once you can predict the common failure modes before they happen, your understanding becomes much more stable.


In [6]:
import dis
import inspect

def probe_runtime(value):
    local_copy = value
    frame = inspect.currentframe()
    print('locals now:', frame.f_locals)
    return local_copy

print(probe_runtime('demo'))
print('\nbytecode for probe_runtime:')
dis.dis(probe_runtime)


locals now: {'value': 'demo', 'local_copy': 'demo', 'frame': <frame at 0x00000261BF257DC0, file 'C:\\Users\\VIHAAN\\AppData\\Local\\Temp\\ipykernel_20016\\939914315.py', line 7, code probe_runtime>}
demo

bytecode for probe_runtime:
  4           0 RESUME                   0

  5           2 LOAD_FAST                0 (value)
              4 STORE_FAST               1 (local_copy)

  6           6 LOAD_GLOBAL              0 (inspect)
             18 LOAD_METHOD              1 (currentframe)
             40 PRECALL                  0
             44 CALL                     0
             54 STORE_FAST               2 (frame)

  7          56 LOAD_GLOBAL              5 (NULL + print)
             68 LOAD_CONST               1 ('locals now:')
             70 LOAD_FAST                2 (frame)
             72 LOAD_ATTR                3 (f_locals)
             82 PRECALL                  2
             86 CALL                     2
             96 POP_TOP

  8          98 LOAD_FAST        

To understand function calls more deeply, inspect the function object itself. The function already stores defaults, constants, variable names, and references to global names before it is ever called. Then each call adds a fresh frame where parameter names are bound to argument objects.


In [7]:

def show_function_runtime_view(fn):
    import dis
    print('function name:', fn.__name__)
    print('varnames:', fn.__code__.co_varnames)
    print('names:', fn.__code__.co_names)
    print('constants:', fn.__code__.co_consts)
    print('freevars:', fn.__code__.co_freevars)
    print('cellvars:', fn.__code__.co_cellvars)
    print('\nbytecode:')
    dis.dis(fn)

def sample(a, b=10, *args, **kwargs):
    local_note = 'inside'
    return a + b

print('defaults:', sample.__defaults__)
print('varnames:', sample.__code__.co_varnames)
print('constants:', sample.__code__.co_consts)
print('names:', sample.__code__.co_names)
show_function_runtime_view(sample)


defaults: (10,)
varnames: ('a', 'b', 'args', 'kwargs', 'local_note')
constants: (None, 'inside')
names: ()
function name: sample
varnames: ('a', 'b', 'args', 'kwargs', 'local_note')
names: ()
constants: (None, 'inside')
freevars: ()
cellvars: ()

bytecode:
 12           0 RESUME                   0

 13           2 LOAD_CONST               1 ('inside')
              4 STORE_FAST               4 (local_note)

 14           6 LOAD_FAST                0 (a)
              8 LOAD_FAST                1 (b)
             10 BINARY_OP                0 (+)
             14 RETURN_VALUE


To understand signatures deeply, think of them as contracts written for both humans and the interpreter. Humans use them to understand intent. Python uses them to bind arguments to parameters in a precise way. This is also why clean signature design matters so much. A good signature reduces ambiguity before the body of the function even begins to run.


## Final Takeaway

The deepest way to revise **Argument Binding, Defaults, and Unpacking Without Hand-Waving** is to stop treating it as a collection of syntax facts and instead treat it as a runtime story. Ask what objects are involved, what names or attributes point to those objects, what frame is active, and what protocol or bytecode path Python is following. If you can explain this topic at those four levels, then you understand much more than the visible syntax.

In interview terms, the strongest answer is usually not the shortest definition. It is a calm explanation of what Python is doing under the surface and why the behavior follows from that model. For revision, come back to this notebook and ask yourself: could I explain this entire chapter with one small code example, one memory-level explanation, and one interpreter-level explanation? If yes, the topic is becoming yours.
